In [1]:
import subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

MAX_ATTEMPTS = 10

def run_notebook_once(notebook):
    path = Path(notebook).resolve()

    print(f'\n===== RUNNING {notebook} =====\n')

    result = subprocess.run([
        'jupyter',
        'nbconvert',
        '--to', 'notebook',
        '--execute',
        '--inplace',
        '--ExecutePreprocessor.timeout=-1',
        path.name,
    ], cwd=path.parent)

    if result.returncode != 0:
        print(f'\nFAILED at {notebook}')
        return False

    print(f'\nFINISHED {notebook}')
    return True

def run_notebook_retry(notebook):
    for attempt in range(1, MAX_ATTEMPTS + 1):
        print(
            f'\n===== ATTEMPT '
            f'{attempt}/{MAX_ATTEMPTS}: '
            f'{notebook} =====\n'
        )

        if run_notebook_once(notebook):
            return True

    return False

def run_parallel(notebooks):
    failed = []

    with ThreadPoolExecutor(
        max_workers=len(notebooks)
    ) as executor:
        futures = {
            executor.submit(
                run_notebook_retry,
                notebook,
            ): notebook
            for notebook in notebooks
        }

        for future in as_completed(futures):
            notebook = futures[future]

            if not future.result():
                failed.append(notebook)

    if not failed:
        return True

    print(
        '\nParallel retries exhausted for: '
        + ', '.join(failed)
    )

    print(
        '\nRetrying failed notebooks '
        'sequentially...\n'
    )

    for notebook in failed:
        if not run_notebook_once(notebook):
            print(f'\nFINAL FAILURE at {notebook}')
            return False

    return True

if not run_parallel([
    '4_INSTRUCTION.ipynb',
    '5_REGISTER.ipynb',
]):
    raise SystemExit(1)

print('\n====')
print('DONE')
print('====')


===== ATTEMPT 1/10: 4_INSTRUCTION.ipynb =====

===== ATTEMPT 1/10: 5_REGISTER.ipynb =====



===== RUNNING 4_INSTRUCTION.ipynb =====


===== RUNNING 5_REGISTER.ipynb =====



[NbConvertApp] Converting notebook 4_INSTRUCTION.ipynb to notebook
[NbConvertApp] Converting notebook 5_REGISTER.ipynb to notebook
[NbConvertApp] Writing 105673 bytes to 4_INSTRUCTION.ipynb



FINISHED 4_INSTRUCTION.ipynb

FINISHED 5_REGISTER.ipynb

====
DONE
====


[NbConvertApp] Writing 211234 bytes to 5_REGISTER.ipynb
